# Stack Overflow tag prediction from titles: ML vs DL proof of concept

<a href="https://colab.research.google.com/github/lrondal/poc-nlp/blob/main/notebooks/stackoverflow_tag_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Goal**: given only the **title** of a Stack Overflow **post**, predict its **tags**.
Two approaches are compared on exactly the same data split and metrics:

| | Approach |
|---|---|
| **ML model** | TF-IDF (word + character n-grams) → One-vs-Rest logistic regression |
| **DL model** | Fine-tuned `distilbert-base-uncased` with a multi-label (sigmoid) head |

Vocabulary used throughout follows [`GLOSSARY.md`](../GLOSSARY.md).

> Run on Google Colab with a GPU runtime (*Runtime → Change runtime type → T4 GPU*).

## 1. Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/lrondal/poc-nlp.git"  # edit if the repo lives elsewhere

if IN_COLAB:
    if not os.path.exists("/content/poc-nlp"):
        !git clone -q {REPO_URL} /content/poc-nlp
    %cd /content/poc-nlp
    # scikit-learn, pandas, torch and transformers are preinstalled on Colab.

In [ ]:
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

SEED = 42
MIN_POSTS_PER_TAG = 20  # a tag becomes a "kept tag" if it appears on at least this many posts

random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = next(p for p in [Path("data/stackoverflow_posts.csv"), Path("../data/stackoverflow_posts.csv")] if p.exists())

## 2. Load the data

The CSV has one row per *(post, tag)* pair. We regroup it into one row per **post** with its **title** and list of **tags**.
`tag_position` is only used to keep the original order; it carries no "main tag" meaning.

In [ ]:
raw = pd.read_csv(DATA_PATH, keep_default_na=False)  # keeps the tag "null" as text
print(f"{len(raw):,} rows, {raw.post_id.nunique():,} posts, {raw.tag_name.nunique():,} distinct tags")
raw.head()

In [ ]:
posts = (
    raw.sort_values(["post_id", "tag_position"])
    .groupby("post_id")
    .agg(title=("title", "first"), tags=("tag_name", list))
    .reset_index()
)
posts.head()

## 3. Exploratory analysis

In [ ]:
tag_counts = posts.explode("tags")["tags"].value_counts()
tags_per_post = posts["tags"].str.len()
title_words = posts["title"].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
tag_counts.head(30).plot.bar(ax=axes[0], title="30 most frequent tags (posts per tag)")
tags_per_post.value_counts().sort_index().plot.bar(ax=axes[1], title="Tags per post")
title_words.plot.hist(bins=30, ax=axes[2], title="Words per title")
plt.tight_layout()
plt.show()

print(f"Tags with >= {MIN_POSTS_PER_TAG} posts: {(tag_counts >= MIN_POSTS_PER_TAG).sum()} / {len(tag_counts)}")
print(f"Tags with exactly 1 post: {(tag_counts == 1).sum()}")
print(f"Longest title: {title_words.max()} words")

The tag distribution is extremely long-tailed: most tags appear on only a handful of posts and cannot be learned
from ~4.6k examples. We therefore restrict prediction to **kept tags**.

## 4. Kept tags, label sets and data split

* **Kept tag**: a tag present on at least `MIN_POSTS_PER_TAG` posts.
* **Label set**: the kept tags of a post. Posts with an empty label set are dropped.
* Split **by post**, 80 / 10 / 10 (train / validation / test), fixed seed. Both models use the exact same split.

In [ ]:
kept_tags = sorted(tag_counts[tag_counts >= MIN_POSTS_PER_TAG].index)
kept = set(kept_tags)

posts["label_set"] = posts["tags"].apply(lambda tags: [t for t in tags if t in kept])
dataset = posts[posts["label_set"].str.len() > 0].reset_index(drop=True)
print(f"{len(kept_tags)} kept tags; {len(dataset):,} / {len(posts):,} posts keep a non-empty label set")

mlb = MultiLabelBinarizer(classes=kept_tags)
Y = mlb.fit_transform(dataset["label_set"]).astype(np.float32)

idx_train, idx_tmp = train_test_split(np.arange(len(dataset)), test_size=0.2, random_state=SEED)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.5, random_state=SEED)

X_train, X_val, X_test = (dataset["title"].iloc[i].tolist() for i in (idx_train, idx_val, idx_test))
Y_train, Y_val, Y_test = Y[idx_train], Y[idx_val], Y[idx_test]
print(f"train={len(X_train)}  val={len(X_val)}  test={len(X_test)}")

### Shared evaluation

Each model outputs a **tag score** in [0, 1] for every kept tag. **Predicted tags** = tags whose score ≥ a threshold,
plus the best-scored tag if nothing passes (every post has at least one tag). The threshold is tuned on the
**validation** set to maximise micro-F1, then frozen for the **test** set.

| Metric | Meaning |
|---|---|
| **micro-F1** | F1 over all (post, tag) decisions: dominated by frequent tags. Headline metric. |
| **macro-F1** | Average of per-tag F1: every kept tag counts equally, so rare tags matter. |
| **P@1** | Is the single best-scored tag in the label set? |
| **R@3** | Share of the label set found among the 3 best-scored tags. |

In [ ]:
def predicted_tags_matrix(scores: np.ndarray, threshold: float) -> np.ndarray:
    pred = (scores >= threshold).astype(int)
    pred[np.arange(len(scores)), scores.argmax(axis=1)] = 1  # always at least one tag
    return pred


def tune_threshold(y_true: np.ndarray, scores: np.ndarray) -> float:
    grid = np.arange(0.05, 0.95, 0.01)
    f1s = [f1_score(y_true, predicted_tags_matrix(scores, t), average="micro", zero_division=0) for t in grid]
    return float(grid[int(np.argmax(f1s))])


def evaluate(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    pred = predicted_tags_matrix(scores, threshold)
    top1 = scores.argmax(axis=1)
    top3 = np.argsort(-scores, axis=1)[:, :3]
    hits_at_3 = np.take_along_axis(y_true, top3, axis=1).sum(axis=1)
    return {
        "micro_f1": f1_score(y_true, pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
        "p@1": y_true[np.arange(len(y_true)), top1].mean(),
        "r@3": (hits_at_3 / y_true.sum(axis=1)).mean(),
        "threshold": threshold,
    }


def top_tags(scores_row: np.ndarray, threshold: float, k: int = 5) -> list[tuple[str, float]]:
    order = np.argsort(-scores_row)[:k]
    chosen = [i for i in order if scores_row[i] >= threshold] or [order[0]]
    return [(kept_tags[i], round(float(scores_row[i]), 3)) for i in chosen]

## 5. ML model: TF-IDF + One-vs-Rest logistic regression

* **Word** n-grams (1–2) with a token pattern that keeps programming names intact (`c#`, `c++`, `.net`, `asp.net`, `ruby-on-rails`).
* **Character** n-grams (2–5) to be robust to spelling variants and unseen words.
* One logistic regression per kept tag. `C` (inverse regularisation) is picked on the validation set.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline, make_union
from sklearn.feature_extraction.text import TfidfVectorizer

TOKEN_PATTERN = r"(?u)\.?[\w#+]+(?:[.\-][\w#+]+)*"


def make_ml_model(C: float):
    features = make_union(
        TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), min_df=2, sublinear_tf=True),
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
    )
    classifier = OneVsRestClassifier(LogisticRegression(C=C, max_iter=2000), n_jobs=-1)
    return make_pipeline(features, classifier)


ml_start = time.perf_counter()
ml_search = {}
for C in [1, 5, 10, 30]:
    candidate = make_ml_model(C).fit(X_train, Y_train)
    val_scores = candidate.predict_proba(X_val)
    ml_search[C] = (evaluate(Y_val, val_scores, tune_threshold(Y_val, val_scores))["micro_f1"], candidate)
    print(f"C={C:<3} val micro-F1={ml_search[C][0]:.3f}")

best_C = max(ml_search, key=lambda c: ml_search[c][0])
ml_model = ml_search[best_C][1]
ml_train_time = time.perf_counter() - ml_start  # includes the C search
print(f"Best C={best_C}; total training time {ml_train_time:.1f}s")

In [ ]:
ml_val_scores = ml_model.predict_proba(X_val)
ml_threshold = tune_threshold(Y_val, ml_val_scores)
ml_test_scores = ml_model.predict_proba(X_test)
ml_results = evaluate(Y_test, ml_test_scores, ml_threshold)
pd.Series(ml_results, name="ML model (test)").round(3)

## 6. DL model: fine-tuned DistilBERT

* `distilbert-base-uncased` pre-trained encoder + a linear layer with one output per kept tag.
* Sigmoid + binary cross-entropy (`problem_type="multi_label_classification"`): each tag is an independent yes/no.
* AdamW with linear warm-up/decay; **early stopping** on validation micro-F1, keeping the best epoch's weights.

In [ ]:
import copy

import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

DL_MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 48        # titles are short; see "Longest title" in section 3
BATCH_SIZE = 32
LEARNING_RATE = 5e-5
MAX_EPOCHS = 20
PATIENCE = 3

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU: this will be slow")

tokenizer = AutoTokenizer.from_pretrained(DL_MODEL_NAME)


def make_loader(titles, labels, shuffle):
    enc = tokenizer(titles, padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")
    ds = TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(labels, dtype=torch.float32))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)


train_loader = make_loader(X_train, Y_train, shuffle=True)
val_loader = make_loader(X_val, Y_val, shuffle=False)
test_loader = make_loader(X_test, Y_test, shuffle=False)

In [ ]:
@torch.no_grad()
def dl_scores(model, loader) -> np.ndarray:
    model.eval()
    out = []
    for input_ids, attention_mask, _ in loader:
        logits = model(input_ids=input_ids.to(device), attention_mask=attention_mask.to(device)).logits
        out.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(out)


dl_model = AutoModelForSequenceClassification.from_pretrained(
    DL_MODEL_NAME,
    num_labels=len(kept_tags),
    problem_type="multi_label_classification",
    id2label=dict(enumerate(kept_tags)),
    label2id={t: i for i, t in enumerate(kept_tags)},
).to(device)

optimizer = torch.optim.AdamW(dl_model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * MAX_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06 * total_steps), total_steps)

history, best_f1, best_state, epochs_without_gain = [], -1.0, None, 0
dl_start = time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    dl_model.train()
    running_loss = 0.0
    for input_ids, attention_mask, labels in train_loader:
        optimizer.zero_grad()
        loss = dl_model(
            input_ids=input_ids.to(device), attention_mask=attention_mask.to(device), labels=labels.to(device)
        ).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(dl_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()

    val_scores = dl_scores(dl_model, val_loader)
    val_f1 = evaluate(Y_val, val_scores, tune_threshold(Y_val, val_scores))["micro_f1"]
    history.append({"epoch": epoch, "train_loss": running_loss / len(train_loader), "val_micro_f1": val_f1})
    print(f"epoch {epoch:2d}  train loss {history[-1]['train_loss']:.4f}  val micro-F1 {val_f1:.3f}")

    if val_f1 > best_f1:
        best_f1, epochs_without_gain = val_f1, 0
        best_state = copy.deepcopy({k: v.detach().cpu() for k, v in dl_model.state_dict().items()})
    else:
        epochs_without_gain += 1
        if epochs_without_gain >= PATIENCE:
            print(f"Early stopping: no improvement for {PATIENCE} epochs")
            break

dl_train_time = time.perf_counter() - dl_start
dl_model.load_state_dict(best_state)
print(f"Best val micro-F1 {best_f1:.3f}; training time {dl_train_time:.1f}s")

In [ ]:
hist = pd.DataFrame(history).set_index("epoch")
fig, ax1 = plt.subplots(figsize=(8, 4))
hist["train_loss"].plot(ax=ax1, color="tab:blue", label="train loss")
ax2 = ax1.twinx()
hist["val_micro_f1"].plot(ax=ax2, color="tab:orange", label="val micro-F1")
ax1.set_ylabel("train loss"); ax2.set_ylabel("val micro-F1")
fig.legend(loc="upper center"); plt.title("DL model training"); plt.show()

In [ ]:
dl_val_scores = dl_scores(dl_model, val_loader)
dl_threshold = tune_threshold(Y_val, dl_val_scores)
dl_test_scores = dl_scores(dl_model, test_loader)
dl_results = evaluate(Y_test, dl_test_scores, dl_threshold)
pd.Series(dl_results, name="DL model (test)").round(3)

## 7. Comparison on the test set

In [ ]:
comparison = pd.DataFrame(
    {
        "ML model (TF-IDF + LogReg)": {**ml_results, "train_time_s": ml_train_time},
        "DL model (DistilBERT)": {**dl_results, "train_time_s": dl_train_time},
    }
).T
comparison.round(3)

In [ ]:
per_tag = pd.DataFrame(
    {
        "posts_in_test": Y_test.sum(axis=0).astype(int),
        "ML_f1": f1_score(Y_test, predicted_tags_matrix(ml_test_scores, ml_threshold), average=None, zero_division=0),
        "DL_f1": f1_score(Y_test, predicted_tags_matrix(dl_test_scores, dl_threshold), average=None, zero_division=0),
    },
    index=kept_tags,
).sort_values("posts_in_test", ascending=False)
per_tag["DL_minus_ML"] = per_tag["DL_f1"] - per_tag["ML_f1"]
per_tag.round(2).head(25)

## 8. Try it: predict tags for any title

In [ ]:
DEMO_TITLES = [
    "How do I parse JSON in Python?",
    "jQuery click event not firing on dynamically added button",
    "NullReferenceException when binding a GridView in ASP.NET",
    "How to join two tables and group by date in MySQL",
    "UITableView cell height not updating on iPhone",
    "Segmentation fault when using pointers to struct in C",
]


def predict_ml(titles):
    return [top_tags(row, ml_threshold) for row in ml_model.predict_proba(titles)]


def predict_dl(titles):
    loader = make_loader(titles, np.zeros((len(titles), len(kept_tags))), shuffle=False)
    return [top_tags(row, dl_threshold) for row in dl_scores(dl_model, loader)]


pd.DataFrame({"title": DEMO_TITLES, "ML model": predict_ml(DEMO_TITLES), "DL model": predict_dl(DEMO_TITLES)})

## 9. Conclusion

In [ ]:
winner = comparison["micro_f1"].idxmax()
gap = comparison["micro_f1"].max() - comparison["micro_f1"].min()
slowdown = dl_train_time / ml_train_time
print(f"Best micro-F1 on test: {winner} (+{gap:.3f})")
print(f"DL training took {slowdown:.1f}x the ML training time (ML time includes the C search)")

**Trade-offs** *(write your observations here after the Colab run, using the table above)*:

* **Quality**: which model wins on micro-F1 (frequent tags) and on macro-F1 (rare tags)? Pre-trained language models
  usually help most on titles that do not name the technology explicitly.
* **Cost**: the ML model trains in seconds on a CPU; the DL model needs a GPU and minutes, and is ~250 MB vs a few MB.
* **Explainability**: the ML model's coefficients show which words drive each tag; the DL model is a black box.
* **Data**: with ~4.6k posts, both are limited to the kept tags; more data would widen the label vocabulary and
  likely benefit the DL model more.

**Next steps towards production**: persist the chosen model, wrap `predict` in an API, monitor tag drift.